In [25]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

In [10]:
def detectar_tipos_de_coluna(df, limiar=0.05):
    categoricas, continuas = [], []
    n = len(df)

    for col in df.columns:
        if df[col].dtype == 'object':
            categoricas.append(col)
        else:
            if df[col].nunique() / n < limiar:
                categoricas.append(col)
            else:
                continuas.append(col)
    return categoricas, continuas

In [11]:
titanic = fetch_openml("titanic", version=1, as_frame=True)
df = titanic.frame[["pclass", "sex", "age", "fare", "survived"]].dropna()

X_df = df.drop("survived", axis=1)
y = df["survived"].astype(int).values


In [20]:
df.head()

,pclass,sex,age,fare,survived
0,1,female,29.0000,211.3375,1
1,1,male,0.9167,151.5500,1
2,1,female,2.0000,151.5500,0
3,1,male,30.0000,151.5500,0
4,1,female,25.0000,151.5500,0


In [ ]:
categorical_cols, continuous_cols = detectar_tipos_de_coluna(X_df)

In [21]:
categorical_cols, continuous_cols

(['pclass', 'sex'], ['age', 'fare'])

In [23]:
for col in categorical_cols:
    X_df[col] = X_df[col].astype(str)

X_cont = X_df[continuous_cols].to_numpy(dtype=float)
X_cat = X_df[categorical_cols].to_numpy(dtype=str)

# Guardar para usar índices depois
idx_cont = list(range(X_cont.shape[1]))
idx_cat = list(range(X_cont.shape[1], X_cont.shape[1] + X_cat.shape[1]))

# Reorganizar matriz final
X = np.concatenate([X_cont, X_cat], axis=1)

X

array([['29.0', '211.3375', '1', 'female'],
       ['0.9167', '151.55', '1', 'male'],
       ['2.0', '151.55', '1', 'female'],
       ...,
       ['26.5', '7.225', '3', 'male'],
       ['27.0', '7.225', '3', 'male'],
       ['29.0', '7.875', '3', 'male']], shape=(1045, 4), dtype='<U32')

In [16]:
class NaiveBayesMisto:
    def __init__(self, idx_cat, idx_cont):
        self.idx_cat = idx_cat
        self.idx_cont = idx_cont
    
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.prior = {}
        self.mean = {}
        self.std = {}
        self.cond_prob = {}

        for c in self.classes:
            X_c = X[y == c]
            self.prior[c] = len(X_c) / len(X)

            cont_vals = X_c[:, self.idx_cont].astype(float)
            self.mean[c] = np.mean(cont_vals, axis=0)
            self.std[c] = np.std(cont_vals, axis=0) + 1e-6  # evitar std zero

            self.cond_prob[c] = {}
            for idx in self.idx_cat:
                valores, cont = np.unique(X_c[:, idx], return_counts=True)
                total = len(X_c)
                self.cond_prob[c][idx] = {v: (cont[i]/total) for i,v in enumerate(valores)}

    def gauss(self, mean, std, x):
        return np.exp(-(x-mean)**2 / (2*std**2)) / np.sqrt(2*np.pi*std**2)

    def predict(self, X):
        preds = []
        for x in X:
            post = {}
            for c in self.classes:
                log_prob = np.log(self.prior[c])

                for i, idx in enumerate(self.idx_cont):
                    log_prob += np.log(self.gauss(self.mean[c][i], self.std[c][i], float(x[idx])))

                for idx in self.idx_cat:
                    v = x[idx]
                    prob = self.cond_prob[c][idx].get(v, 1e-9)
                    log_prob += np.log(prob)

                post[c] = log_prob
            preds.append(max(post, key=post.get))
        return preds


In [24]:
nb = NaiveBayesMisto(idx_cat, idx_cont)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

X_train.shape, X_test.shape


((731, 4), (314, 4))

In [18]:
nb.fit(X_train, y_train)
preds = nb.predict(X_test)

In [26]:
recall = recall_score(y_test, preds)
precision = precision_score(y_test, preds)
f1_score = f1_score(y_test, preds)

In [27]:
print("Matriz de Confusão:\n", confusion_matrix(y_test, preds))
print("Acurácia:", accuracy_score(y_test, preds))
print("Recall:", recall)
print("Precision:", precision)
print("F1 Score:", f1_score)

Matriz de Confusão:
 [[159  16]
 [ 65  74]]
Acurácia: 0.7420382165605095
Recall: 0.5323741007194245
Precision: 0.8222222222222222
F1 Score: 0.6462882096069869
